# Step 2 — OCT on the midtrained bases

Step 1 (`midtrain.ipynb`) produced one midtrained base per (student, constitution) and pushed
it to HF. This runs standard OCT on top of each, following `run_olmo.sh`: one constitution per
GPU lane, `OCT_MASTER_PORT` offset per lane, logs to `/workspace/lane{N}.log`, resume by
re-running (finished stages are skipped).

In [ ]:
import os, json, pathlib, subprocess

os.environ["HF_HOME"]     = "/workspace/.cache/huggingface"
os.environ["HF_TOKEN"]    = ""
os.environ["WANDB_TOKEN"] = ""
os.environ["HF_USER"]     = "invi-bhagyesh"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # as in run_olmo.sh

# must match midtrain.ipynb exactly -- these names key the weights dir, the data dirs and the repos
STUDENTS_TO_RUN = ["qwen", "olmo"]
CONSTITUTIONS   = ["goodness", "sycophancy"]
TEACHER_ID      = "deepseek/deepseek-v4-pro"
TEACHER         = TEACHER_ID.split("/")[-1]

STUDENTS = {
    "qwen": dict(hf_id="Qwen/Qwen2.5-7B-Instruct",   local="qwen-2.5-7b-it",     extra=[]),
    "olmo": dict(hf_id="allenai/OLMo-2-1124-7B-SFT", local="olmo-2-1124-7b-sft", extra=["--gradient_checkpointing"]),
}
HF_USER    = os.environ["HF_USER"]
WORKSPACE  = "/workspace"
OCT        = f"{WORKSPACE}/OpenCharacterTraining"
MODELS_DIR = f"{WORKSPACE}/models"

def names(student, cons):
    S = STUDENTS[student]
    return dict(student=student, cons=cons, S=S,
                local = f"{S['local']}-msm-{TEACHER}-{cons}",   # midtrained base == run_data's $M
                key   = f"{student}_msm_{TEACHER}_{cons}")      # run_all.py MODELS entry

PAIRS = [(s, c) for s in STUDENTS_TO_RUN for c in CONSTITUTIONS]

def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=os.environ)
    if r.returncode: raise RuntimeError(f"exit {r.returncode}: {cmd}")

import torch
NG = torch.cuda.device_count()
print(f"{len(PAIRS)} pairs, {NG} GPUs\n")
for i, (s, c) in enumerate(PAIRS):
    n = names(s, c)
    print(f"  lane{i % max(NG,1)}  {s:5s} {c:11s}  M={n['local']}  key={n['key']}")

## 1. Pull the midtrained bases

Step 1 may have run on a different pod. `run_data.py` and `run_all.py` both resolve models as
`{MODEL_PATH}/{name}`, so each base has to be on disk under its own `local_name`.

In [ ]:
from huggingface_hub import snapshot_download

for s, c in PAIRS:
    n = names(s, c); dest = f"{MODELS_DIR}/{n['local']}"
    if os.path.exists(f"{dest}/config.json"):
        print(f"{n['local']}: on disk"); continue
    print(f"{n['local']}: downloading...")
    snapshot_download(repo_id=f"{HF_USER}/{n['local']}", local_dir=dest,
                      token=os.environ["HF_TOKEN"])
    print(f"  -> {dest}")

## 2. Register each pair in `run_all.py`

`run_all.py` resolves weights *and* data from `cfg['local_name']`, so each midtrained base
needs its own entry. `extra_args` is copied from run_all's own table — olmo needs
`--gradient_checkpointing` or DPO OOMs on an 80GB card.

No data symlinks: OCT's DPO `rejected` responses and its introspection data are generated by
the student itself, and the student here is the midtrained model. `run_data.py` regenerates
both against `$M` below.

In [ ]:
run_all = pathlib.Path(f"{OCT}/run_all.py"); text = run_all.read_text()

for s, c in PAIRS:
    n = names(s, c)
    if f'"{n["key"]}"' in text:
        print(f"{n['key']}: already registered"); continue
    extra = ", ".join(f'"{a}"' for a in n["S"]["extra"])
    entry = (f'MODELS = {{\n'
             f'    "{n["key"]}": {{\n'
             f'        "hf_id": "{n["S"]["hf_id"]}",\n'
             f'        "local_name": "{n["local"]}",\n'
             f'        "dpo_micro_batch": 1,\n'
             f'        "sft_micro_batch": 1,\n'
             f'        "extra_args": [{extra}],\n'
             f'    }},\n')
    assert text.count("MODELS = {") == 1
    text = text.replace("MODELS = {", entry, 1)
    print(f"{n['key']}: registered -> {n['local']}")
run_all.write_text(text)

## 3. Run OCT

One pair per GPU lane, exactly the `run_olmo.sh` sequence with `$M` bound to the midtrained
base and `--model` to its registered key:

```
run_data.py --stage dpo --model $M   ->  seeds shared teacher responses, then vLLM for $M's own
run_all.py  --stage dpo              ->  DPO LoRA
run_all.py  --stage fold             ->  merge into the base
run_data.py --stage sft --model $M   ->  introspection, generated from the folded model
run_all.py  --stage sft              ->  SFT LoRA
tools/upload_data.py                 ->  persist it; the pod is ephemeral
```

In [ ]:
lanes = {}
for i, (s, c) in enumerate(PAIRS):
    lanes.setdefault(i % max(NG, 1), []).append(names(s, c))

script = ["#!/bin/bash", "cd /workspace/OpenCharacterTraining", "source .env 2>/dev/null",
          f'export HF_USER={HF_USER}', 'export HF_HOME=/workspace/.cache/huggingface',
          'export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True', "",
          "run_one () {   # $1=gpu  $2=M (midtrained base)  $3=key  $4=constitution",
          "  export CUDA_VISIBLE_DEVICES=$1",
          "  export OCT_MASTER_PORT=$((29500 + $1))",
          '  echo "=== [lane$1] START $4 on $2 $(date -u +%H:%M) ==="',
          "  python3 run_data.py --stage dpo --model $2 --constitution $4 \\",
          "    && python3 run_all.py  --model $3 --constitution $4 --stage dpo \\",
          "    && python3 run_all.py  --model $3 --constitution $4 --stage fold \\",
          "    && python3 run_data.py --stage sft --model $2 --constitution $4 \\",
          "    && python3 run_all.py  --model $3 --constitution $4 --stage sft \\",
          "    && python3 tools/upload_data.py --model $2 --constitution $4 \\",
          '    && echo "=== [lane$1] DONE $4 $(date -u +%H:%M) ===" \\',
          '    || echo "=== [lane$1] FAILED $4 ==="', "}", ""]

for lane, items in sorted(lanes.items()):
    body = "; ".join(f'run_one {lane} {n["local"]} {n["key"]} {n["cons"]}' for n in items)
    script.append(f"( {body} ) > /workspace/lane{lane}.log 2>&1 &")
script += ["wait", 'echo "ALL LANES DONE"']

path = f"{OCT}/run_msm_oct.sh"
pathlib.Path(path).write_text("\n".join(script) + "\n")
print(pathlib.Path(path).read_text())

In [ ]:
# Runs to completion; each lane's output is in /workspace/lane{N}.log.
# Re-running is safe: every stage skips work that already exists.
sh("bash run_msm_oct.sh", cwd=OCT)

In [ ]:
# watch progress from another cell while the above runs
sh("tail -n 15 /workspace/lane*.log")

## 4. Merge and push the final models

`run_all.py` pushes the DPO and SFT **LoRAs** to `{HF_USER}/{local_name}-{constitution}` but
never a merged checkpoint. This is the standalone model to load for evals.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

for s, c in PAIRS:
    n = names(s, c)
    distilled = f"{MODELS_DIR}/distilled/{n['local']}-{c}"
    sft_lora  = f"{WORKSPACE}/loras/{n['key']}-introspection/{c}"
    final_dir = f"{MODELS_DIR}/final/{n['local']}-{c}"
    if not (os.path.exists(distilled) and os.path.exists(sft_lora)):
        print(f"{s}/{c}: incomplete, skipping"); continue

    m = AutoModelForCausalLM.from_pretrained(distilled, torch_dtype=torch.bfloat16, device_map="cpu")
    m = PeftModel.from_pretrained(m, sft_lora).merge_and_unload()
    os.makedirs(final_dir, exist_ok=True)
    m.save_pretrained(final_dir)
    AutoTokenizer.from_pretrained(distilled).save_pretrained(final_dir)
    del m

    pathlib.Path(final_dir, "README.md").write_text(
        f"# Condition B final — {s} / {c}\n\n"
        f"- **teacher (wrote the corpus)**: `{TEACHER_ID}`\n"
        f"- **student**: `{n['S']['hf_id']}`\n"
        f"- **constitution**: `{c}`\n"
        f"- **stage**: `MSM midtraining -> OCT DPO (folded) -> OCT SFT (folded)`\n"
        f"- **midtrained base**: `{HF_USER}/{n['local']}`\n"
        f"- **LoRAs**: `{HF_USER}/{n['local']}-{c}`\n"
        f"- **OCT data**: `regenerated against the midtrained model`\n")

    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    repo = f"{HF_USER}/{n['local']}-{c}-final"
    api.create_repo(repo_id=repo, exist_ok=True, private=True)
    api.upload_folder(folder_path=final_dir, repo_id=repo)
    print(f"pushed -> https://huggingface.co/{repo}")